In [1]:
import sys


In [2]:
sys.path.append("/user/christoph.wald/u15287/insect_pest_detection/modules")

In [9]:
import modules

In [8]:
modules.draw_box

<function modules.draw_box(img, box, color, label)>

In [10]:
from modules import *

In [ ]:
draw_box()

TypeError: draw_box() missing 4 required positional arguments: 'img', 'box', 'color', and 'label'

In [2]:
!!python3 -V


['Python 3.11.9']

In [2]:
import os

def files_are_identical(file1, file2, chunk_size=4096):
    # First, check if both files exist
    if not os.path.exists(file1) or not os.path.exists(file2):
        return False
    
    # Check file size first (fast check)
    if os.path.getsize(file1) != os.path.getsize(file2):
        return False

    # Compare file content in chunks to handle large files efficiently
    with open(file1, 'rb') as f1, open(file2, 'rb') as f2:
        while True:
            b1 = f1.read(chunk_size)
            b2 = f2.read(chunk_size)
            if b1 != b2:
                return False
            if not b1:  # end of file
                break

    return True


# Example usage:
file1 = "/user/christoph.wald/u15287/insect_pest_detection/2_5_self_training/modules.py"
file2 = "/user/christoph.wald/u15287/insect_pest_detection/modules/modules.py"

if files_are_identical(file1, file2):
    print("✅ The files are identical.")
else:
    print("❌ The files are different.")


❌ The files are different.


## Test pseudo label weighing
### Setup a small test set

In [6]:
import os
import shutil

def copy_folder_limited_sorted(src, dst, file_limit=5):
    """
    Copy a folder and its subfolders, but only the first `file_limit` files
    in each folder, sorted alphabetically.
    
    :param src: Source directory path
    :param dst: Destination directory path
    :param file_limit: Number of files to copy per folder
    """
    if not os.path.exists(dst):
        os.makedirs(dst)

    for root, dirs, files in os.walk(src):
        # Sort files alphabetically
        files.sort()
        
        # Calculate relative path from source folder
        rel_path = os.path.relpath(root, src)
        dest_path = os.path.join(dst, rel_path)
        
        # Create destination subfolder if it doesn't exist
        if not os.path.exists(dest_path):
            os.makedirs(dest_path)
        
        # Copy only the first `file_limit` files
        for file in files[:file_limit]:
            src_file = os.path.join(root, file)
            dst_file = os.path.join(dest_path, file)
            shutil.copy2(src_file, dst_file)

# Example usage:
copy_folder_limited_sorted("/user/christoph.wald/u15287/big-scratch/04_SSL_training_data/training_data", "/user/christoph.wald/u15287/big-scratch/04_SSL_training_data/test_data")


### Test yolo

In [1]:
from ultralytics import YOLO

Here we are (DetectionTrainer)


In [2]:
model = YOLO('yolov8s.pt')
model.train(data='/user/christoph.wald/u15287/big-scratch/04_SSL_training_data/test_data/data.yaml', 
            epochs=1,
            patience = 10, 
            imgsz=640,
            
            scale=0.3, #instead of 0.5
            mosaic= 0.25, #instead of 1.0
            mixup=0.05, #instead of 0.0
            erasing=0.4, #default (increase when oberving false positives)
            auto_augment="randaugment", #default, maybe try augmix
            
            crop_fraction= 0.1, #(heavy cropping!) instead of 1.0
            multi_scale= True,
            fliplr = 0.3, #instead fo 0.5
            cache = False
            
            
            )
print("Finished.")

Here we are!
New https://pypi.org/project/ultralytics/8.3.209 available 😃 Update with 'pip install -U ultralytics'
WARNING ⚠️ 'crop_fraction' is deprecated and will be removed in the future.
Ultralytics 8.3.207 🚀 Python-3.11.9 torch-2.8.0+cu128 CPU (Intel Xeon Silver 4210 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/user/christoph.wald/u15287/big-scratch/04_SSL_training_data/test_data/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.3, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_rati

KeyboardInterrupt: 

In [ ]:
import os

def append_float_to_txt_lines(folder, value_to_add):
    """
    Appends a specified float to each line of all .txt files in a folder.
    
    :param folder: Path to the folder containing .txt files
    :param value_to_add: Float value to append as text to each line
    """
    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder, filename)
            
            # Read original lines
            with open(file_path, 'r') as f:
                lines = f.readlines()
            
            # Append the float to each line
            new_lines = [line.rstrip('\n') + f" {value_to_add}\n" for line in lines]
            
            # Overwrite the file with updated lines
            with open(file_path, 'w') as f:
                f.writelines(new_lines)

# Example usage:
# append_float_to_txt_lines("my_folder", 1.5)


# Example usage:
#append_float_to_txt_lines("/user/christoph.wald/u15287/big-scratch/04_SSL_training_data/test_data/labels/train", 0.5)
append_float_to_txt_lines("/user/christoph.wald/u15287/big-scratch/04_SSL_training_data/test_data/labels/val", 0.75)


In [4]:
from ultralytics.yolo.data.dataset import YOLODataset
dataset = YOLODataset(data ='/user/christoph.wald/u15287/big-scratch/04_SSL_training_data/test_data/data.yaml',  )

ModuleNotFoundError: No module named 'ultralytics.yolo'